# 📈 Snorkel Intro Tutorial: Data Augmentation

In this tutorial, we will walk through the process of using *transformation functions* (TFs) to perform data augmentation.
Like the labeling tutorial, our goal is to train a classifier to YouTube comments as `SPAM` or `HAM` (not spam).
In the [previous tutorial](https://github.com/snorkel-team/snorkel-tutorials/blob/master/spam/01_spam_tutorial.ipynb),
we demonstrated how to label training sets programmatically with Snorkel.
In this tutorial, we'll assume that step has already been done, and start with labeled training data,
which we'll aim to augment using transformation functions.


Data augmentation is a popular technique for increasing the size of labeled training sets by applying class-preserving transformations to create copies of labeled data points.
In the image domain, it is a crucial factor in almost every state-of-the-art result today and is quickly gaining
popularity in text-based applications.
Snorkel models the data augmentation process by applying user-defined *transformation functions* (TFs) in sequence.
You can learn more about data augmentation in
[this blog post about our NeurIPS 2017 work on automatically learned data augmentation](https://snorkel.org/blog/tanda/).

The tutorial is divided into four parts:
1. **Loading Data**: We load a [YouTube comments dataset](http://www.dt.fee.unicamp.br/~tiago//youtubespamcollection/).
2. **Writing Transformation Functions**: We write Transformation Functions (TFs) that can be applied to training data points to generate new training data points.
3. **Applying Transformation Functions to Augment Our Dataset**: We apply a sequence of TFs to each training data point, using a random policy, to generate an augmented training set.
4. **Training a Model**: We use the augmented training set to train an LSTM model for classifying new comments as `SPAM` or `HAM`.

This next cell takes care of some notebook-specific housekeeping.
You can ignore it.

In [1]:
import os
import random

import numpy as np

# Turn off TensorFlow logging messages
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# For reproducibility
seed = 0
os.environ["PYTHONHASHSEED"] = str(seed)
np.random.seed(0)
random.seed(0)

If you want to display all comment text untruncated, change `DISPLAY_ALL_TEXT` to `True` below.

In [2]:
import pandas as pd


DISPLAY_ALL_TEXT = True

pd.set_option("display.max_colwidth", 0 if DISPLAY_ALL_TEXT else 50)

This next cell makes sure a spaCy English model is downloaded.
If this is your first time downloading this model, restart the kernel after executing the next cell.

In [3]:
# Download the spaCy english model
# ! python -m spacy download en_core_web_sm

## 1. Loading Data

We load the Kaggle dataset and create Pandas DataFrame objects for the `train` and `test` sets.
The two main columns in the DataFrames are:
* **`text`**: Raw text content of the comment
* **`label`**: Whether the comment is `SPAM` (1) or `HAM` (0).

For more details, check out the [labeling tutorial](https://github.com/snorkel-team/snorkel-tutorials/blob/master/spam/01_spam_tutorial.ipynb).

In [4]:
from utils import load_spam_dataset

df_train, df_test = load_spam_dataset(load_train_labels=True)

# We pull out the label vectors for ease of use later
Y_train = df_train["label"].values
Y_test = df_test["label"].values

In [5]:
df_train.head()

,author,date,text,label,video
,,,,,
,,,,,
,,,,,
,,,,,
,,,,,


## 2. Writing Transformation Functions (TFs)

Transformation functions are functions that can be applied to a training data point to create another valid training data point of the same class.
For example, for image classification problems, it is common to rotate or crop images in the training data to create new training inputs.
Transformation functions should be atomic e.g. a small rotation of an image, or changing a single word in a sentence.
We then compose multiple transformation functions when applying them to training data points.

Common ways to augment text includes replacing words with their synonyms, or replacing names entities with other entities.
More info can be found
[here](https://towardsdatascience.com/data-augmentation-in-nlp-2801a34dfc28) or
[here](https://towardsdatascience.com/these-are-the-easiest-data-augmentation-techniques-in-natural-language-processing-you-can-think-of-88e393fd610).
Our basic modeling assumption is that applying these operations to a comment generally shouldn't change whether it is `SPAM` or not.

Transformation functions in Snorkel are created with the
[`transformation_function` decorator](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.transformation_function.html#snorkel.augmentation.transformation_function),
which wraps a function that takes in a single data point and returns a transformed version of the data point.
If no transformation is possible, a TF can return `None` or the original data point.
If all the TFs applied to a data point return `None`, the data point won't be included in
the augmented dataset when we apply our TFs below.

Just like the `labeling_function` decorator, the `transformation_function` decorator
accepts `pre` argument for `Preprocessor` objects.
Here, we'll use a
[`SpacyPreprocessor`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/preprocess/snorkel.preprocess.nlp.SpacyPreprocessor.html#snorkel.preprocess.nlp.SpacyPreprocessor).

In [6]:
from snorkel.preprocess.nlp import SpacyPreprocessor

spacy = SpacyPreprocessor(text_field="text", doc_field="doc", memoize=True)

In [7]:
import names
from snorkel.augmentation import transformation_function

# Pregenerate some random person names to replace existing ones with
# for the transformation strategies below
replacement_names = [names.get_full_name() for _ in range(50)]


# Replace a random named entity with a different entity of the same type.
@transformation_function(pre=[spacy])
def change_person(x):
    person_names = [ent.text for ent in x.doc.ents if ent.label_ == "PERSON"]
    # If there is at least one person name, replace a random one. Else return None.
    if person_names:
        name_to_replace = np.random.choice(person_names)
        replacement_name = np.random.choice(replacement_names)
        x.text = x.text.replace(name_to_replace, replacement_name)
        return x


# Swap two adjectives at random.
@transformation_function(pre=[spacy])
def swap_adjectives(x):
    adjective_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "ADJ"] # pos: parts-of-speech
    # Check that there are at least two adjectives to swap.
    if len(adjective_idxs) >= 2:
        idx1, idx2 = sorted(np.random.choice(adjective_idxs, 2, replace=False))
        # Swap tokens in positions idx1 and idx2.
        x.text = " ".join(
            [
                x.doc[:idx1].text,
                x.doc[idx2].text,
                x.doc[1 + idx1 : idx2].text,
                x.doc[idx1].text,
                x.doc[1 + idx2 :].text,
            ]
        )
        return x

We add some transformation functions that use `wordnet` from [NLTK](https://www.nltk.org/) to replace different parts of speech with their synonyms.

In [8]:
import nltk
from nltk.corpus import wordnet as wn

nltk.download("wordnet")


def get_synonym(word, pos=None):
    """Get synonym for word given its part-of-speech (pos)."""
    synsets = wn.synsets(word, pos=pos)
    # Return None if wordnet has no synsets (synonym sets) for this word and pos.
    if synsets:
        words = [lemma.name() for lemma in synsets[0].lemmas()]
        if words[0].lower() != word.lower():  # Skip if synonym is same as word.
            # Multi word synonyms in wordnet use '_' as a separator e.g. reckon_with. Replace it with space.
            return words[0].replace("_", " ")


def replace_token(spacy_doc, idx, replacement):
    """Replace token in position idx with replacement."""
    return " ".join([spacy_doc[:idx].text, replacement, spacy_doc[1 + idx :].text])


@transformation_function(pre=[spacy])
def replace_verb_with_synonym(x):
    # Get indices of verb tokens in sentence.
    verb_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "VERB"]
    if verb_idxs:
        # Pick random verb idx to replace.
        idx = np.random.choice(verb_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="v")
        # If there's a valid verb synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x


@transformation_function(pre=[spacy])
def replace_noun_with_synonym(x):
    # Get indices of noun tokens in sentence.
    noun_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "NOUN"]
    if noun_idxs:
        # Pick random noun idx to replace.
        idx = np.random.choice(noun_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="n")
        # If there's a valid noun synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x


@transformation_function(pre=[spacy])
def replace_adjective_with_synonym(x):
    # Get indices of adjective tokens in sentence.
    adjective_idxs = [i for i, token in enumerate(x.doc) if token.pos_ == "ADJ"]
    if adjective_idxs:
        # Pick random adjective idx to replace.
        idx = np.random.choice(adjective_idxs)
        synonym = get_synonym(x.doc[idx].text, pos="a")
        # If there's a valid adjective synonym, replace it. Otherwise, return None.
        if synonym:
            x.text = replace_token(x.doc, idx, synonym)
            return x

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/vaishnavihemaljariwala/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [9]:
tfs = [
    change_person,
    swap_adjectives,
    replace_verb_with_synonym,
    replace_noun_with_synonym,
    replace_adjective_with_synonym,
]

Let's check out a few examples of transformed data points to see what our TFs are doing.

In [10]:
from utils import preview_tfs

preview_tfs(df_train, tfs)

,TF Name,Original Text,Transformed Text
0,,,
1,,,
2,,,
3,,,
4,,,


We notice a couple of things about the TFs.

* Sometimes they make trivial changes (`"website"` to `"web site"` for replace_noun_with_synonym).
  This can still be helpful for training our model, because it teaches the model to be invariant to such small changes.
* Sometimes they introduce incorrect grammar to the sentence (e.g. `swap_adjectives` swapping `"young"` and `"more"` above).

The TFs are expected to be heuristic strategies that indeed preserve the class most of the time, but
[don't need to be perfect](https://arxiv.org/pdf/1901.11196.pdf).
This is especially true when using automated
[data augmentation techniques](https://snorkel.org/blog/tanda/)
which can learn to avoid particularly corrupted data points.
As we'll see below, Snorkel is compatible with such learned augmentation policies.

## 3. Applying Transformation Functions

We'll first define a `Policy` to determine what sequence of TFs to apply to each data point.
We'll start with a [`RandomPolicy`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.RandomPolicy.html)
that samples `sequence_length=2` TFs to apply uniformly at random per data point.
The `n_per_original` argument determines how many augmented data points to generate per original data point.

In [11]:
from snorkel.augmentation import RandomPolicy

random_policy = RandomPolicy(
    len(tfs), sequence_length=2, n_per_original=2, keep_original=True
)

In some cases, we can do better than uniform random sampling.
We might have domain knowledge that some TFs should be applied more frequently than others,
or have trained an [automated data augmentation model](https://snorkel.org/blog/tanda/)
that learned a sampling distribution for the TFs.
Snorkel supports this use case with a
[`MeanFieldPolicy`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.MeanFieldPolicy.html),
which allows you to specify a sampling distribution for the TFs.
We give higher probabilities to the `replace_[X]_with_synonym` TFs, since those provide more information to the model.

In [12]:
from snorkel.augmentation import MeanFieldPolicy

mean_field_policy = MeanFieldPolicy(
    len(tfs),
    sequence_length=2,
    n_per_original=2,
    keep_original=True,
    p=[0.05, 0.05, 0.3, 0.3, 0.3],
)

To apply one or more TFs that we've written to a collection of data points according to our policy, we use a
[`PandasTFApplier`](https://snorkel.readthedocs.io/en/master/packages/_autosummary/augmentation/snorkel.augmentation.PandasTFApplier.html)
because our data points are represented with a Pandas DataFrame.

In [13]:
from snorkel.augmentation import PandasTFApplier

tf_applier = PandasTFApplier(tfs, mean_field_policy)
df_train_augmented = tf_applier.apply(df_train)
Y_train_augmented = df_train_augmented["label"].values

  0%|          | 0/1586 [00:00<?, ?it/s]

  2%|▏         | 25/1586 [00:00<00:06, 244.73it/s]

  3%|▎         | 50/1586 [00:00<00:06, 242.05it/s]

  5%|▍         | 75/1586 [00:00<00:06, 242.43it/s]

  6%|▋         | 100/1586 [00:00<00:06, 235.38it/s]

  8%|▊         | 124/1586 [00:00<00:06, 229.44it/s]

... (57 outputs left out)

 98%|█████████▊| 1554/1586 [00:07<00:00, 81.84it/s]

 99%|█████████▊| 1563/1586 [00:08<00:00, 74.87it/s]

100%|█████████▉| 1584/1586 [00:08<00:00, 98.13it/s]

100%|██████████| 1586/1586 [00:08<00:00, 191.51it/s]

In [14]:
print(f"Original training set size: {len(df_train)}")
print(f"Augmented training set size: {len(df_train_augmented)}")

Original training set size: 1586
Augmented training set size: 2428


We have almost doubled our dataset using TFs!
Note that despite `n_per_original` being set to 2, our dataset may not exactly triple in size,
because sometimes TFs return `None` instead of a new data point
(e.g. `change_person` when applied to a sentence with no persons).
If you prefer to have exact proportions for your dataset, you can have TFs that can't perform a
valid transformation return the original data point rather than `None` (as they do here).

## 4. Training A Model

Our final step is to use the augmented data to train a model. We train an LSTM (Long Short Term Memory) model, which is a very standard architecture for text processing tasks.

The next cell makes Keras results reproducible. You can ignore it.

In [15]:
import tensorflow as tf

# The TF1-style Session setup was removed in Keras 3.
# This sets the Python, NumPy and TensorFlow random seeds in one call, for reproducibility.
tf.keras.utils.set_random_seed(0)

Now we'll train our LSTM on both the original and augmented datasets to compare performance.

In [16]:
from utils import featurize_df_tokens, get_keras_lstm

X_train = featurize_df_tokens(df_train)
X_train_augmented = featurize_df_tokens(df_train_augmented)
X_test = featurize_df_tokens(df_test)


def train_and_test(X_train, Y_train, X_test=X_test, Y_test=Y_test, num_buckets=30000):
    # Define a vanilla LSTM model with Keras
    lstm_model = get_keras_lstm(num_buckets)
    lstm_model.fit(X_train, Y_train, epochs=5, verbose=0)
    preds_test = lstm_model.predict(X_test)[:, 0] > 0.5
    return (preds_test == Y_test).mean()


acc_augmented = train_and_test(X_train_augmented, Y_train_augmented)
acc_original = train_and_test(X_train, Y_train)

1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


1/8 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


In [17]:
print(f"Test Accuracy (original training data): {100 * acc_original:.1f}%")
print(f"Test Accuracy (augmented training data): {100 * acc_augmented:.1f}%")

Test Accuracy (original training data): 47.2%
Test Accuracy (augmented training data): 71.6%


So using the augmented dataset indeed improved our model!
There is a lot more you can do with data augmentation, so try a few ideas
out on your own!

---
# Experiments

**Compatibility fix:** the original "make Keras reproducible" cell used `tf.compat.v1.Session` and `set_session`, which Keras 3 removed. It was replaced with `tf.keras.utils.set_random_seed(0)`. That still fixes the random seeds but no longer forces single-threaded execution, so LSTM results can vary slightly between runs.

**Baseline observation:** the test set has 118 SPAM and 132 HAM comments, and 118 / 250 = **47.2%**, exactly the accuracy reported for the model trained on the original data. So that model predicts SPAM for every comment: it hasn't learned anything. The original tutorial output shows the same 47.2% / 71.6%, so this comes from the tutorial setup itself, not from our environment.

In [18]:
def train_and_test_v2(X_train, Y_train, epochs=5, X_test=X_test, Y_test=Y_test, num_buckets=30000):
    """Same as train_and_test, but epochs is configurable, the seed is reset before every run
    (so each model starts from the same weights), and it also returns the share of SPAM predictions."""
    tf.keras.utils.set_random_seed(0)
    lstm_model = get_keras_lstm(num_buckets)
    lstm_model.fit(X_train, Y_train, epochs=epochs, verbose=0)
    preds_test = lstm_model.predict(X_test, verbose=0)[:, 0] > 0.5
    return (preds_test == Y_test).mean(), preds_test.mean()


print(f"Test labels: {(Y_test == 1).sum()} SPAM / {(Y_test == 0).sum()} HAM")

Test labels: 118 SPAM / 132 HAM


### Experiment 1: Train for more epochs

**Change:** Train the same LSTM for 20 epochs instead of 5, on both the original and the augmented data. Also report how often each model predicts SPAM.

**Why:** With 5 epochs the model trained on the original data never gets past "always predict SPAM". The augmented set is about 1.5x larger, so in 5 epochs it gets about 1.5x more weight updates. That alone could explain its higher accuracy, without the augmentation itself being useful. If both models are trained long enough to actually learn, the comparison is fair.

In [19]:
exp1_results = {}
for epochs in [5, 20]:
    for name, X, Y in [("original", X_train, Y_train), ("augmented", X_train_augmented, Y_train_augmented)]:
        acc, spam_rate = train_and_test_v2(X, Y, epochs=epochs)
        exp1_results[(epochs, name)] = acc
        print(f"epochs={epochs:>2}  {name:<9}  test acc = {100 * acc:.1f}%   predicted SPAM = {100 * spam_rate:.0f}%")

epochs= 5  original   test acc = 47.2%   predicted SPAM = 100%


epochs= 5  augmented  test acc = 71.6%   predicted SPAM = 20%


epochs=20  original   test acc = 47.2%   predicted SPAM = 100%


epochs=20  augmented  test acc = 72.0%   predicted SPAM = 21%


**Results (Experiment 1):**

| Training data | Epochs | Test acc. | Predicted SPAM |
|---|---|---|---|
| Original | 5 | 47.2% | 100% |
| Augmented | 5 | 71.6% | 20% |
| Original | 20 | 47.2% | 100% |
| Augmented | 20 | 72.0% | 21% |

**Takeaway:** The hypothesis was wrong. Even with 4x more training, the model trained on the original data still predicts SPAM for every comment. It never escapes that stuck state, so the problem is not simply too few weight updates. Training the augmented model longer barely changed it (+0.4, which is 1 test comment).

So more epochs alone don't explain the 47% vs 72% gap. But the "always SPAM" predictions show that the baseline model isn't learning at all. That makes "augmentation improved accuracy" a comparison against a broken baseline, not evidence that the augmented examples are useful. Experiment 2 looks into this further.

### Experiment 2: Add a random word deletion transformation function

**Change:** Added a new TF, `delete_random_word`, which drops one random word from comments with at least 4 words. It joins the five existing TFs in a new mean-field policy. The new TF gets probability 0.3 and the three synonym TFs go from 0.3 to 0.2 each, so the probabilities still sum to 1. Models are trained for 20 epochs (from Experiment 1) and compared with Experiment 1's 20-epoch augmented result.

**Why:** All five existing TFs need spaCy to find a person name, adjectives or a word with a WordNet synonym, so many comments can't be transformed at all. Random word deletion is a standard text augmentation ("Easy Data Augmentation") that works on almost any comment. It teaches the model not to rely on any single word. The risk is that it sometimes deletes the very word that makes a comment spam, such as "subscribe", which creates a slightly wrongly labelled example.

In [20]:
@transformation_function()
def delete_random_word(x):
    words = x.text.split()
    # Only transform comments long enough to survive losing a word. Else return None.
    if len(words) < 4:
        return None
    del words[np.random.randint(len(words))]
    x.text = " ".join(words)
    return x


tfs_v2 = tfs + [delete_random_word]
preview_tfs(df_train, [delete_random_word])

,TF Name,Original Text,Transformed Text
0,,,


In [21]:
np.random.seed(0)
random.seed(0)

mean_field_policy_v2 = MeanFieldPolicy(
    len(tfs_v2),
    sequence_length=2,
    n_per_original=2,
    keep_original=True,
    p=[0.05, 0.05, 0.2, 0.2, 0.2, 0.3],
)
df_train_augmented_v2 = PandasTFApplier(tfs_v2, mean_field_policy_v2).apply(df_train)
Y_train_augmented_v2 = df_train_augmented_v2["label"].values
X_train_augmented_v2 = featurize_df_tokens(df_train_augmented_v2)

print(f"Augmented training set size (original TFs):         {len(df_train_augmented)}")
print(f"Augmented training set size (+ delete_random_word): {len(df_train_augmented_v2)}")

acc_v2, spam_rate_v2 = train_and_test_v2(X_train_augmented_v2, Y_train_augmented_v2, epochs=20)
print(f"\nepochs=20  original              test acc = {100 * exp1_results[(20, 'original')]:.1f}%")
print(f"epochs=20  augmented (5 TFs)     test acc = {100 * exp1_results[(20, 'augmented')]:.1f}%")
print(f"epochs=20  augmented (+ delete)  test acc = {100 * acc_v2:.1f}%   predicted SPAM = {100 * spam_rate_v2:.0f}%")

  0%|          | 0/1586 [00:00<?, ?it/s]

  3%|▎         | 53/1586 [00:00<00:02, 529.66it/s]

  8%|▊         | 128/1586 [00:00<00:02, 649.42it/s]

 12%|█▏        | 193/1586 [00:00<00:02, 573.73it/s]

 17%|█▋        | 275/1586 [00:00<00:01, 661.81it/s]

 22%|██▏       | 355/1586 [00:00<00:01, 707.12it/s]

... (24 outputs left out)

 99%|█████████▉| 1572/1586 [00:04<00:00, 118.99it/s]

100%|██████████| 1586/1586 [00:04<00:00, 318.46it/s]

Augmented training set size (original TFs):         2428
Augmented training set size (+ delete_random_word): 3323



epochs=20  original              test acc = 47.2%
epochs=20  augmented (5 TFs)     test acc = 72.0%
epochs=20  augmented (+ delete)  test acc = 47.2%   predicted SPAM = 100%


**Results (Experiment 2):**

| Training data (20 epochs) | Training set size | Test acc. | Predicted SPAM |
|---|---|---|---|
| Original | 1,586 | 47.2% | 100% |
| Augmented, 5 original TFs | 2,428 | 72.0% | 21% |
| Augmented, + `delete_random_word` | 3,323 | 47.2% | 100% |

**Takeaway:** `delete_random_word` worked as a transformation. It applies to far more comments than the spaCy-based TFs, so the augmented set grew to 3,323 comments (from 2,428). But the model trained on it collapsed back to predicting SPAM for every comment.

Read together with Experiment 1, this points to the model setup, not the data, as the main problem. The tutorial's LSTM (`get_keras_lstm` in `utils.py`) uses the Adagrad optimizer with its small default learning rate and a ReLU activation inside the LSTM. With that setup, training often gets stuck at "always predict the majority class": SPAM is the majority in every training set here (52%). Whether a given run escapes seems to depend on the exact data, not on whether the data is better. More data was not enough, and neither was more epochs.

**Overall conclusion for this notebook:** the tutorial's "augmentation improved the model" result does not hold up. The augmented model also under-predicts SPAM (21% predicted vs 47% actual), so even its 72% is not a strong model. Augmentation can only be judged fairly once the base model trains reliably, for example after switching the optimizer to Adam. Each result here also comes from a single seed, so the numbers should be read as indicative only.